データセット内のファイル一覧を取得

In [4]:
from huggingface_hub import list_repo_files

repo_id = "L-FAME-Dataset-Benchmark/L-FAME"

files = list_repo_files(
    repo_id=repo_id,
    repo_type="dataset"
)

print("ファイル総数:", len(files))

for f in files[:100]:
    print(f)

ファイル総数: 3442
.gitattributes
README.md
croissant.json
dataset_api.py
dataset_description.json
derivatives/eeglab_preproc/dataset_description.json
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task

In [5]:
targets = [
    "restCE01",
    "Medita",
    "restCE02",
    "slMedita"
]

selected_files = []

for f in files:

    if "derivatives/eeglab_preproc/" not in f:
        continue

    if not any(task in f for task in targets):
        continue

    selected_files.append(f)

print("対象ファイル数:", len(selected_files))

for f in selected_files[:100]:
    print(f)

対象ファイル数: 892
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-slMedita_eeg_prepro

- 最終的にクリーニングされた _icrm を取る

In [6]:
from collections import Counter
from pathlib import Path

ext_counts = Counter(Path(f).suffix for f in selected_files)

print(ext_counts)

Counter({'.set': 652, '.fdt': 240})


sub-001のみ表示

In [7]:
for f in selected_files:
    if "sub-001" in f:
        print(f)

derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-slMedita_eeg_preproc_icrm.set
de

selected_files をさらに絞る

In [8]:
final_files = []

for f in selected_files:

    # restCE01 → ICA除去済み
    if "task-restCE01" in f and "preproc_icrm.set" in f:
        final_files.append(f)

    # restCE02 → ICA除去済み
    elif "task-restCE02" in f and "preproc_icrm.set" in f:
        final_files.append(f)

    # slMedita → ICA除去済み
    elif "task-slMedita" in f and "preproc_icrm.set" in f:
        final_files.append(f)

    # Medita(avtive meditation) → preica の set + fdt
    elif "task-Medita" in f and "preproc_preica" in f:
        if f.endswith(".set") or f.endswith(".fdt"):
            final_files.append(f)


print("最終ダウンロード対象:", len(final_files))

for f in final_files[:30]:
    print(f)

最終ダウンロード対象: 358
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-slMedita_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-premedita/eeg/sub-001_ses-premedita_task-Medita_eeg_preproc_preica.fdt
derivatives/eeglab_preproc/sub-001/ses-premedita/eeg/sub-001_ses-premedita_task-Medita_eeg_preproc_preica.set
derivatives/eeglab_preproc/sub-001/ses-premedita/eeg/sub-001_ses-premedita_task-restCE01_eeg_preproc_icrm.set
derivatives/eeglab_preproc/sub-001/ses-premedita/eeg/sub-001_ses-premedita_task-restCE02_eeg_preproc_icr

Medita を解析対象にするなら、基本的には自分で論文内のノイズ除去まで行う必要がある

一応Medita もダウンロード
- restCE01 → icrm.set
- restCE02 → icrm.set
- slMedita → icrm.set
- Medita → preica.set + preica.fdt

In [9]:
from huggingface_hub import hf_hub_download
from pathlib import Path

repo_id = "L-FAME-Dataset-Benchmark/L-FAME"
# data/内に保存
save_dir = Path("data")

print("最終ダウンロード対象:", len(final_files))

for i, f in enumerate(final_files, start=1):

    print(f"[{i}/{len(final_files)}] {f}")

    hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=f,
        # data/の中にfのディレクトリで保存する
        local_dir=save_dir
    )

print("ダウンロード完了")

最終ダウンロード対象: 358
[1/358] derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.fdt
[2/358] derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-Medita_eeg_preproc_preica.set
[3/358] derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE01_eeg_preproc_icrm.set
[4/358] derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-restCE02_eeg_preproc_icrm.set


[5/358] derivatives/eeglab_preproc/sub-001/ses-posmedita/eeg/sub-001_ses-posmedita_task-slMedita_eeg_preproc_icrm.set
[6/358] derivatives/eeglab_preproc/sub-001/ses-premedita/eeg/sub-001_ses-premedita_task-Medita_eeg_preproc_preica.fdt
[7/358] derivatives/eeglab_preproc/sub-001/ses-premedita/eeg/sub-001_ses-premedita_task-Medita_eeg_preproc_preica.set
[8/358] derivatives/eeglab_preproc/sub-001/ses-premedita/eeg/sub-001_ses-premedita_task-restCE01_eeg_preproc_icrm.set
[9/358] derivatives/eeglab_preproc/sub-001/ses-premedita/eeg/sub-001_ses-premedita_task-restCE02_eeg_preproc_icrm.set
[10/358] derivatives/eeglab_preproc/sub-001/ses-premedita/eeg/sub-001_ses-premedita_task-slMedita_eeg_preproc_icrm.set
[11/358] derivatives/eeglab_preproc/sub-002/ses-posmedita/eeg/sub-002_ses-posmedita_task-Medita_eeg_preproc_preica.fdt
[12/358] derivatives/eeglab_preproc/sub-002/ses-posmedita/eeg/sub-002_ses-posmedita_task-Medita_eeg_preproc_preica.set
[13/358] derivatives/eeglab_preproc/sub-002/ses-posme